In [396]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Imports

In [397]:
# general imports
import scipy
import os
import re
import mne
import numpy as np
import yasa
from scipy.signal import hilbert
import sys

# import from custom script in same directory
import shared_processing_functions as spf

# import from different directory
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from hdf5_files import Artefacts_Detection as ad

## Paths

In [398]:
# root directory for all data
main_data_path = r"D:\dilon_data\edf_collection\main"
#main_data_path = r"D:\dilon_data\datasets\MOTORWP4_dataset\test"
# annotation directories
sleep_edf_anno = r"D:\dilon_data\annotation_collection\Sleep-EDF"
motorwp4_anno = r"D:\dilon_data\annotation_collection\MOTORWP4"
# output path
output_path = r"D:\dilon_data\mat_collection\static"

## variables

In [399]:
# regex pattern to extract subject
pattern = re.compile(r'(SC4\d{5}|ST7\d{5}|S\d{2,3}_\d)')
# wake time (s) to save before first sleep and after last sleep
# (30 mins, so 30(s) * 60(s))
wake_time = 1800

# which channels to extract
channels = ["Fpz-Cz", "Pz-Oz", "horizontal", "submental"]

# stage names and corresponding ids
sleep_edf_stage_id = {
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
    "Movement time": 5,
    "Sleep stage ?": 6
}

motorwp4_stage_id = {'0':0,'1':1,'2':2,'3':3,'5':4}

# target bands to use for psd
target_bands = ['Noise', 'Delta', 'Theta', 'Sigma', 'Gamma']
# # stage to use for psd

# power band frequencies to use
bands_list = [
    [0, 0.5, 'Noise'],
    [0.5, 4, 'Delta'],
    [5, 11, 'Theta'],
    [11, 17, 'Sigma'],
    [35, 45, 'Gamma']
]

# Main

### Functions

#### Find all edf files

In [400]:
def find_edf(path):
    raw_files = []
    for file in os.listdir(os.path.join(path)):
        if not file.startswith('.'):
            if ".edf" in file:
                raw_files.append(file)
    
    return raw_files

#### Add anotation to raw object

In [401]:
def add_annotation(anno_file, raw):
    # load mat file with annotation
    mat_data = scipy.io.loadmat(anno_file)
    states = mat_data["states"]

    # get values from 2d array
    descriptions = [str(int(s[0])) for s in states]  # state labels as strings
    onsets = [float(s[2]) for s in states]           # onset in seconds
    durations = [float(s[3]) for s in states]        # duration in seconds

    # create annotations object
    annotations = mne.Annotations(onset=onsets, duration=durations, description=descriptions)

    # set annotations
    raw.set_annotations(annotations)

#### crop data to annotation size

In [402]:
def crop_to_anno(raw):
    segments = []
    for onset, duration in zip(raw.annotations.onset, raw.annotations.duration):
        onset_rel = onset -raw.first_time
        tmin = onset_rel
        tmax = min(onset_rel + duration, raw.times[-1])

        # Skip invalid chunks
        if tmin >= tmax:
            continue

        seg = raw.copy().crop(tmin=tmin, tmax=tmax)  # lazy if preload=False
        segments.append(seg)

    raw_cropped = mne.concatenate_raws(segments)
    
    return raw_cropped

#### PSD

In [403]:
def get_power_band(data, bands):
    sf = data.info['sfreq']
    raw_eeg = data.copy().pick(['Fpz-Cz', 'Pz-Oz'])

    # Compute bandpower using YASA (returns DataFrame)
    bp_df = yasa.bandpower(raw_eeg, sf=sf, bands=bands)
    bp_df

    return bp_df

#### Channel with highest power

In [404]:
def best_channel(df, band):
    # get electrode with highest power
    max_channel = df[band].idxmax()

    return max_channel

#### RMS smoothing

In [405]:
def rms(signal, window_size):
    signal_sq = np.square(signal)
    window = np.ones(window_size)/float(window_size)
    rms = np.sqrt(np.convolve(signal_sq, window, 'same'))

    return rms

#### Create EMG channel

In [406]:
def create_emg_channel(mne_obj, high_pass, low_pass, window_size):
    # 4th order butterworth bandpass filter
    mne_obj.filter(
        l_freq=high_pass,
        h_freq=low_pass,
        method="iir",
        iir_params=dict(order=4, ftype="butter")
    )

    # split the two channels
    left_channel = mne_obj.copy().pick(["E240"]).get_data()
    right_channel = mne_obj.copy().pick(["E243"]).get_data()

    left_channel, _, _ = ad.removeArtefacts(left_channel[0], 250, [9,8], [0.2,0.1])
    right_channel, _, _ = ad.removeArtefacts(right_channel[0], 250, [9,8], [0.2,0.1])

    # left_channel = np.abs(left_channel)
    # right_channel = np.abs(right_channel)

    # # apply Hilbert transform
    # left_channel = np.abs(hilbert(left_channel))
    # right_channel = np.abs(hilbert(right_channel))

    left_channel = rms(np.squeeze(left_channel), window_size)
    right_channel = rms(np.squeeze(right_channel), window_size)

    combined_emg = (left_channel + right_channel) / 2

    return combined_emg

#### Create EOG channel

In [407]:
def create_eog_channel(mne_obj, high_pass, low_pass):
    # 4th order butterworth bandpass filter
    mne_obj.filter(
        l_freq=high_pass,
        h_freq=low_pass,
        method="iir",
        iir_params=dict(order=4, ftype="butter")
    )

    # split the two channels
    left_channel = mne_obj.copy().pick(["E10"]).get_data()
    right_channel = mne_obj.copy().pick(["E54"]).get_data()
    left_channel, _, _ = ad.removeArtefacts(left_channel[0], 250, [5,4], [0.2,0.1])
    right_channel, _, _ = ad.removeArtefacts(right_channel[0], 250, [5,4], [0.2,0.1])

    # apply Hilbert transform
    left_channel = np.abs(hilbert(left_channel))
    right_channel = np.abs(hilbert(right_channel))

    bipolar_eog = left_channel - right_channel

    return bipolar_eog

### Workflow

#### Get list with all edf files

In [408]:
# find all edf files
edf_files = find_edf(main_data_path)

print(f"Amount of files available: {len(edf_files)}")

Amount of files available: 33


In [ ]:
for edf_file_path in edf_files:
    # dictionary to save the different bands with highest 
    # power in specific channel
    channel_bands = {}
    # create directory to save info to
    final_results = {}

    # find subject
    match = pattern.search(edf_file_path)
    if match:
        old_subject = match.group(1)

    # check if subject name is in the right format
    if "_" not in old_subject:
        subject = old_subject[:-1] + "_" + old_subject[-1]
    else:
        subject = old_subject

    # skips subjects that already exists
    if any(subject in file for file in list(os.listdir(output_path))):
        continue
    else:
        print(f'Extracting data from subject: {subject}')
        print('-' * 50)

        # read raw data
        raw = mne.io.read_raw_edf(
            os.path.join(main_data_path, edf_file_path), 
            channels, 
            infer_types=True
        )

        if "SC" in subject or "ST" in subject:
            # set correct sampling frequency
            fs = 100
            # find correct annotation file
            anno_file = next((
                anno for anno in os.listdir(sleep_edf_anno) 
                if old_subject in anno), 
                None
            )

            # extract and annotate raw data
            spf.add_annotation(os.path.join(sleep_edf_anno, anno_file), raw)

        else:
            # set correct sampling frequency
            fs = 250
            # find correct annotation file (different filetype)
            anno_file = next((
                anno for anno in os.listdir(motorwp4_anno) 
                if subject in anno and ".mat" in anno), 
                None
            )
            print(anno_file)
            # annotate raw
            add_annotation(os.path.join(motorwp4_anno, anno_file), raw)

        ### cropping data
        # check if raw data is longer than annotation
        if raw.times[-1] > raw.annotations.duration.sum():
            # crop raw to annotation
            temp_raw = crop_to_anno(raw)
        else:
            temp_raw = raw

        # crop the raw data
        cropped_raw = spf.crop_data(temp_raw, wake_time)

        # load data for additional filters and processing
        cropped_raw.load_data()
        
        #cropped_raw.set_channel_types({"Fpz-Cz": "eeg", "Pz-Oz": "eeg"})

        ### create .mat file for the annotation
        # get sleep states from cropped raw and save to .mat file
        if "SC" in subject or "ST" in subject:
            sleep_states = spf.get_stages(cropped_raw, sleep_edf_stage_id)
            spf.create_mat(output_path, subject, "states", sleep_states)
        else:
            sleep_states = spf.get_stages(cropped_raw, motorwp4_stage_id)
            spf.create_mat(output_path, subject, "states", sleep_states)

        #     # bandpass
        #     cropped_raw.filter(l_freq=0.25, h_freq=45, picks=best_ch)
        # ### get powerbands
        # bp_mean = get_power_band(cropped_raw, bands_list)

        # ### get the best channel for each power band and save data in dict
        # for i in range(len(target_bands)):
        #     # get the best channel
        #     try:
        #         best_ch = best_channel(bp_mean, target_bands[i])
        #     except:
        #         best_ch = best_channel(bp_mean, target_bands[i])

        #     cropped_raw.apply_function(lambda x: mne.filter.detrend(x, axis=0, order=1), picks=best_ch)
        #     # bandpass
        #     cropped_raw.filter(l_freq=0.25, h_freq=45, picks=best_ch)

        #     # get data from best channel
        #     ch_data = cropped_raw.get_data(best_ch)
        #     ch_data, _, _ = ad.removeArtefacts(ch_data[0], 250, [9,8], [0.2,0.1])
        #     # add channel to dictionary
        #     if best_ch not in final_results:
        #         final_results[best_ch] = [target_bands[i], ch_data]
        #     else:
        #         final_results[best_ch][0] += f"_{target_bands[i]}"

        # ### save data from dict to .mat file
        # for channel, ch_values in final_results.items():
        #     ch_pb = f"{channel}_" + ch_values[0]
        #     spf.create_mat(output_path, subject, ch_pb, ch_values[1])

        cropped_raw.apply_function(lambda x: mne.filter.detrend(x, axis=0, order=1), picks=["Fpz-Cz", "Pz-Oz"])
        cropped_raw.filter(l_freq=0.1, h_freq=45, picks=["Fpz-Cz", "Pz-Oz"])

        if "SC" in subject or "ST" in subject:
            for channel in ['horizontal', 'submental']:
                cropped_raw_c = cropped_raw.copy().pick([channel])
                # rename indistinct channel names
                if channel == "horizontal":
                    cropped_raw_c.rename_channels({"horizontal": "EOG"})
                    channel = "EOG"
                elif channel == "submental":
                    cropped_raw_c.rename_channels({"submental":"EMG"})
                    channel = "EMG"

                cropped_data = cropped_raw_c.get_data()
                spf.create_mat(output_path, subject, channel, cropped_data)
        else:
            for channel in ["Fpz-Cz", "Pz-Oz"]:
                cropped_raw_c = cropped_raw.copy().pick([channel])
                cropped_data = cropped_raw_c.get_data()
                cropped_data, _, _ = ad.removeArtefacts(cropped_data[0], 250, [5,4], [0.2,0.1])
                spf.create_mat(output_path, subject, channel, cropped_data)
            emg_picks = cropped_raw.copy().pick(["E240", "E243"])
            emg_picks.load_data()
            # 50 sample window for 200 ms time frame
            combined_emg = create_emg_channel(emg_picks, 5, 120, fs) 
            spf.create_mat(output_path, subject, "EMG", combined_emg)

            eog_picks = cropped_raw.copy().pick(["E10", "E54"])
            eog_picks.load_data()
            bipolar_eog = create_eog_channel(eog_picks, 0.5, 35)
            spf.create_mat(output_path, subject, "EOG", bipolar_eog)

Extracting data from subject: S40_2
--------------------------------------------------
Extracting EDF parameters from D:\dilon_data\edf_collection\main\MOTORWP4_S40_2.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
MOTORWP4_S40_2 20210412 23_states.mat
First sleep starts at: 0.0
Last sleep ends at: 30870.368
Cropping raw: 0 - 30870.368
Cropping finished.
Reading 0 ... 7717592  =      0.000 ... 30870.368 secs...
Sleep stages variable can't be reshaped.
Saving the sleep stages to .mat file.
Filtering a subset of channels. The highpass and lowpass values in the measurement info will not be updated.
Filtering raw data in 93 contiguous segments
Setting up band-pass filter from 0.1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower 